# Pipeline Orchestration (Source -> Bronze -> Silver -> Gold)

This notebook orchestrates the full metadata-driven pipeline by calling the other notebooks in order:
1. **metadata_config** — create/refresh metadata control tables
2. **data_ingestion** — ingest source data into bronze
3. **data_transformation** — transform bronze into silver
4. **data_quality** — validate silver tables against DQ rules
5. **Gold layer** — create fact tables, dimension tables (incl. date dimension), and business aggregates

It also generates a pipeline execution summary from `pipeline_run_log`.

In [0]:
# ============================================================================
# Configuration
# ============================================================================
CATALOG = "metadata_framework"
META_SCHEMA = f"{CATALOG}.bronze"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

# Pipeline notebook paths (relative to the folder)
PIPELINE_FOLDER = "/Users/2016bec008@sggs.ac.in/metadata driven pipeline"
NOTEBOOKS = {
    "metadata_config": f"{PIPELINE_FOLDER}/metadata_config",
    "data_ingestion": f"{PIPELINE_FOLDER}/data_ingestion",
    "data_transformation": f"{PIPELINE_FOLDER}/data_transformation",
    "data_quality": f"{PIPELINE_FOLDER}/data_quality",
}

print(f"Pipeline orchestration configured.")
print(f"Catalog: {CATALOG}")
print(f"Notebooks to execute: {list(NOTEBOOKS.keys())}")

In [0]:
# ============================================================================
# Step 1: Run metadata_config (create/refresh control tables)
# ============================================================================
print("\n" + "="*60)
print("STEP 1: Metadata Configuration")
print("="*60)
try:
    dbutils.notebook.run(NOTEBOOKS["metadata_config"], 300)
    print("✅ metadata_config completed successfully")
except Exception as e:
    print(f"❌ metadata_config failed: {str(e)[:500]}")

In [0]:
# ============================================================================
# Step 2: Run data_ingestion (source → bronze)
# ============================================================================
print("\n" + "="*60)
print("STEP 2: Data Ingestion (Source → Bronze)")
print("="*60)
try:
    dbutils.notebook.run(NOTEBOOKS["data_ingestion"], 600)
    print("✅ data_ingestion completed successfully")
except Exception as e:
    print(f"❌ data_ingestion failed: {str(e)[:500]}")

In [0]:
# ============================================================================
# Step 3: Run data_transformation (bronze → silver)
# ============================================================================
print("\n" + "="*60)
print("STEP 3: Data Transformation (Bronze → Silver)")
print("="*60)
try:
    dbutils.notebook.run(NOTEBOOKS["data_transformation"], 600)
    print("✅ data_transformation completed successfully")
except Exception as e:
    print(f"❌ data_transformation failed: {str(e)[:500]}")

In [0]:
# ============================================================================
# Step 4: Run data_quality (validate silver)
# ============================================================================
print("\n" + "="*60)
print("STEP 4: Data Quality Validation")
print("="*60)
try:
    dbutils.notebook.run(NOTEBOOKS["data_quality"], 300)
    print("✅ data_quality completed successfully")
except Exception as e:
    print(f"❌ data_quality failed: {str(e)[:500]}")

In [0]:
# ============================================================================
# Step 5: Gold Layer — Star Schema (Dimensions + Facts + Aggregates)
# ============================================================================
print("\n" + "="*60)
print("STEP 5: Gold Layer (Silver -> Gold)")
print("="*60)

GOLD = f"{CATALOG}.gold"
SILVER = f"{CATALOG}.silver"

try:
    # ---- 1. dim_customers ----
    spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.dim_customers
    AS
    SELECT
        customer_id, customer_name, email, phone, address, city, country,
        registration_date, bronze_ingested_at, silver_transformed_at,
        current_timestamp() AS gold_processed_at
    FROM {SILVER}.customers
    """)
    print(f"  dim_customers: {spark.read.table(f'{GOLD}.dim_customers').count()} rows")

    # ---- 2. dim_products ----
    spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.dim_products
    AS
    SELECT
        product_id, product_name, category, unit_price, cost_price,
        (unit_price - cost_price) AS margin,
        stock_quantity, supplier, launch_date,
        bronze_ingested_at, silver_transformed_at,
        current_timestamp() AS gold_processed_at
    FROM {SILVER}.products
    """)
    print(f"  dim_products: {spark.read.table(f'{GOLD}.dim_products').count()} rows")

    # ---- 3. dim_date (2022-01-01 to 2025-12-31) ----
    spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.dim_date
    AS
    SELECT
        d AS date_id, d AS full_date,
        YEAR(d) AS year, QUARTER(d) AS quarter, MONTH(d) AS month,
        MONTHNAME(d) AS month_name, DAY(d) AS day_of_month,
        DAYOFWEEK(d) AS day_of_week, DAYNAME(d) AS day_name,
        WEEKOFYEAR(d) AS week_of_year, DAYOFYEAR(d) AS day_of_year,
        CASE WHEN MONTH(d) IN (12,1,2) THEN 'Winter'
             WHEN MONTH(d) IN (3,4,5) THEN 'Spring'
             WHEN MONTH(d) IN (6,7,8) THEN 'Summer'
             ELSE 'Fall' END AS season,
        CASE WHEN MONTH(d) IN (1,2,3) THEN 'Q1'
             WHEN MONTH(d) IN (4,5,6) THEN 'Q2'
             WHEN MONTH(d) IN (7,8,9) THEN 'Q3'
             ELSE 'Q4' END AS quarter_label,
        CONCAT(YEAR(d), '-', LPAD(CAST(MONTH(d) AS STRING), 2, '0')) AS year_month,
        CASE WHEN DAYOFWEEK(d) IN (1,7) THEN true ELSE false END AS is_weekend
    FROM (SELECT explode(sequence(
        CAST('2022-01-01' AS DATE), CAST('2025-12-31' AS DATE), INTERVAL 1 DAY
    )) AS d) t
    """)
    print(f"  dim_date: {spark.read.table(f'{GOLD}.dim_date').count()} rows")

    # ---- 4. fact_orders ----
    spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.fact_orders
    AS
    SELECT
        o.order_id, o.customer_id, o.product_id, o.order_date AS order_date_id,
        o.quantity, o.unit_price, o.total_amount,
        (o.quantity * o.unit_price) AS calculated_total, o.order_status,
        p.product_name, p.category, c.customer_name, c.city, c.country,
        o.bronze_ingested_at, o.silver_transformed_at,
        current_timestamp() AS gold_processed_at
    FROM {SILVER}.orders o
    LEFT JOIN {SILVER}.customers c ON o.customer_id = c.customer_id
    LEFT JOIN {SILVER}.products p ON o.product_id = p.product_id
    """)
    print(f"  fact_orders: {spark.read.table(f'{GOLD}.fact_orders').count()} rows")

    # ---- 5. fact_payments ----
    spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.fact_payments
    AS
    SELECT
        pm.payment_id, pm.order_id, pm.customer_id, pm.payment_date AS payment_date_id,
        pm.payment_method, pm.payment_amount, pm.payment_status, pm.transaction_ref,
        o.order_date, o.total_amount AS order_amount,
        c.customer_name, c.city, c.country,
        pm.bronze_ingested_at, pm.silver_transformed_at,
        current_timestamp() AS gold_processed_at
    FROM {SILVER}.payments pm
    LEFT JOIN {SILVER}.orders o ON pm.order_id = o.order_id
    LEFT JOIN {SILVER}.customers c ON pm.customer_id = c.customer_id
    """)
    print(f"  fact_payments: {spark.read.table(f'{GOLD}.fact_payments').count()} rows")

    # ---- 6. agg_customer_order_summary ----
    spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.agg_customer_order_summary
    AS
    SELECT
        c.customer_id, c.customer_name, c.email, c.city, c.country,
        COUNT(DISTINCT o.order_id) AS total_orders,
        SUM(o.total_amount) AS total_revenue,
        AVG(o.total_amount) AS avg_order_value,
        MIN(o.order_date) AS first_order_date,
        MAX(o.order_date) AS last_order_date,
        COUNT(DISTINCT pm.payment_id) AS total_payments,
        SUM(pm.payment_amount) AS total_paid,
        current_timestamp() AS gold_processed_at
    FROM {SILVER}.customers c
    LEFT JOIN {SILVER}.orders o ON c.customer_id = o.customer_id
    LEFT JOIN {SILVER}.payments pm ON o.order_id = pm.order_id
    GROUP BY c.customer_id, c.customer_name, c.email, c.city, c.country
    """)
    print(f"  agg_customer_order_summary: {spark.read.table(f'{GOLD}.agg_customer_order_summary').count()} rows")

    # ---- 7. agg_product_sales_summary ----
    spark.sql(f"""
    CREATE OR REPLACE TABLE {GOLD}.agg_product_sales_summary
    AS
    SELECT
        p.product_id, p.product_name, p.category, p.unit_price, p.cost_price,
        (p.unit_price - p.cost_price) AS margin,
        COUNT(o.order_id) AS total_orders,
        SUM(o.quantity) AS total_quantity_sold,
        SUM(o.total_amount) AS total_revenue,
        SUM(o.total_amount * ((p.unit_price - p.cost_price) / p.unit_price)) AS total_margin,
        current_timestamp() AS gold_processed_at
    FROM {SILVER}.products p
    LEFT JOIN {SILVER}.orders o ON p.product_id = o.product_id
    GROUP BY p.product_id, p.product_name, p.category, p.unit_price, p.cost_price
    """)
    print(f"  agg_product_sales_summary: {spark.read.table(f'{GOLD}.agg_product_sales_summary').count()} rows")

    print("\nGold layer complete: 3 dimensions, 2 facts, 2 aggregates")
    print("\n--- fact_orders sample ---")
    display(spark.read.table(f"{GOLD}.fact_orders").limit(5))
    print("\n--- dim_date sample ---")
    display(spark.read.table(f"{GOLD}.dim_date").limit(5))

except Exception as e:
    print(f"Gold layer failed: {str(e)[:500]}")

In [0]:
# ============================================================================
# Pipeline Execution Summary
# ============================================================================
print("\n" + "="*60)
print("PIPELINE EXECUTION SUMMARY")
print("="*60)

# Read pipeline run log for this execution window
run_log = spark.read.table(f"{META_SCHEMA}.pipeline_run_log")
display(run_log.orderBy("started_at", ascending=False).limit(20))

# DQ summary (guard against table not existing yet)
try:
    dq_results = spark.read.table(f"{META_SCHEMA}.dq_results")
    print("\nData Quality Results:")
    display(dq_results.orderBy("checked_at", ascending=False).limit(20))
except Exception:
    print("\nNo DQ results table found yet.")

# Counts
total_runs = run_log.count()
success_runs = run_log.filter("run_status = 'success'").count()
failed_runs = run_log.filter("run_status = 'failed'").count()

print(f"\nTotal runs logged: {total_runs}")
print(f"  Successful: {success_runs}")
print(f"  Failed: {failed_runs}")
print(f"\nPipeline orchestration complete.")